# Analisis aktivitas petir tahunan pada 617 tower

Notebook ini menjelaskan cleaning, pembentukan fitur, pemilihan model, evaluasi retrospektif 2025, dan keselarasan dengan perhitungan PLN. `Count` adalah jumlah petir teramati di area sekitar tower, **bukan** sambaran langsung ke struktur atau jumlah gangguan transmisi. Jalankan seluruh sel dari atas ke bawah dengan workbook asli tersedia lokal.

## 1. Pertanyaan analisis dan definisi
### 1.1 Pertanyaan
Seberapa baik riwayat petir, lokasi, dan atribut tower menjelaskan `Count` tahun berikutnya? Bagaimana hasil pada skala paparan tahunan dibandingkan dengan `P_strike_annual` workbook PLN?

### 1.2 Batas makna
Model menebak `Count` pada area pengamatan. `Density` adalah normalisasi dari `Count`; `P_strike_annual` adalah hasil rumus PLN yang turut memakai geometri dan faktor eksposur. Workbook PLN memakai kepadatan petir 2025, sehingga perbandingan di sini **bukan adu dua ramalan independen**.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook

from ta_ml.cleaning import (
    RAW_GEOMETRY_FILENAME, RAW_LIGHTNING_FILENAME, RAW_STATIC_FILENAME,
    build_clean_workbook,
)
from ta_ml.config import COUNT_DENSITY_DIVISOR, TOWER_COUNT
from ta_ml.evaluation import evaluate_models
from ta_ml.panel import build_model_panel

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get('TA_ML_DATA_DIR', ROOT / 'data')).resolve()
OUTPUT_DIR = Path(os.environ.get('TA_ML_OUTPUT_DIR', ROOT / 'data' / 'output')).resolve()
required_files = [RAW_LIGHTNING_FILENAME, RAW_STATIC_FILENAME, RAW_GEOMETRY_FILENAME]
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Workbook asli belum tersedia di {DATA_DIR}: {missing}')
print('Data lokal:', DATA_DIR)
print('Hasil lokal:', OUTPUT_DIR)

## 2. Data mentah sebelum cleaning
### 2.1 Sumber dan sheet yang dipakai
Tiga workbook berikut tetap read-only. Jumlah baris sheet ditampilkan untuk menunjukkan sumber yang diperiksa, bukan dianggap otomatis valid.

In [ ]:
sources = [
    (RAW_LIGHTNING_FILENAME, 'PKY-DATA PETIR', 'riwayat Count 2020–2025'),
    (RAW_STATIC_FILENAME, 'Kasongan-SKS', 'contoh sheet snapshot atribut 2026'),
    (RAW_GEOMETRY_FILENAME, 'INPUT_TOWER', 'geometri dan input PLN'),
]
source_rows = []
for filename, sheet_name, role in sources:
    workbook = load_workbook(DATA_DIR / filename, read_only=True, data_only=True)
    try:
        source_rows.append({'file': filename, 'sheet': sheet_name, 'baris_sheet': workbook[sheet_name].max_row, 'peran': role})
    finally:
        workbook.close()
display(pd.DataFrame(source_rows))

## 3. Cleaning dan validasi data
### 3.1 Aturan dan keluaran cleaning
Cleaning dijalankan oleh modul Python agar aturan yang sama dipakai oleh notebook, CLI, dan tes. Excel mentah tidak ditulis ulang. Workbook hasil menyimpan tabel tower–tahun, registry tower, audit koordinat 2020, dan catatan sumber. Tanda sumber seperti `-` dan `Terendam Air` tidak dipaksa menjadi angka.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean = build_clean_workbook(
    source_dir=DATA_DIR, output_path=OUTPUT_DIR / 'ta_ml_clean_v2.xlsx', overwrite=True
)
history = pd.read_excel(clean.output_path, sheet_name='tower_year')
registry = pd.read_excel(clean.output_path, sheet_name='tower_registry')
audit_2020 = pd.read_excel(clean.output_path, sheet_name='coordinate_2020_audit')
source_notes = pd.read_excel(clean.output_path, sheet_name='source_notes')
print(f'{len(history):,} tower–tahun; {len(registry)} tower dalam registry')
display(source_notes[['source', 'time_scope', 'join_or_usage_rule']])

### 3.2 Audit nama dan GPS 2020: sebelum–sesudah
Nama sumber 2020 dipertahankan dalam audit. Hanya koordinat yang cocok secara unik yang boleh menetapkan identitas tower clean; delapan nilai yang belum pasti tetap tidak ditugaskan.

In [ ]:
display(audit_2020['match_status'].value_counts().rename_axis('status').reset_index(name='baris'))
display(audit_2020[['source_tower_key', 'matched_tower_key', 'source_count_2020', 'match_status']].head(12))
coverage = history.groupby('year').agg(baris=('tower_key', 'size'), count_terisi=('count', 'count'), tower_unik=('tower_key', 'nunique')).reset_index()
display(coverage)
assert len(history) == TOWER_COUNT * 6 and len(registry) == TOWER_COUNT
assert not history.duplicated(['corridor', 'tower_key', 'year']).any()
assert (coverage['baris'] == TOWER_COUNT).all()

### 3.3 Nilai, geometri, dan provenance
Periksa bahwa atribut tower tersambung pada identitas yang benar. Hubungan `Count`–`Density` diperiksa sebagai fakta data; kolom `Area` sumber tidak diam-diam ditafsirkan sebagai pembagi karena nilainya berbeda dari pembagi empiris.

In [ ]:
valid_density = history.dropna(subset=['count', 'density'])
density_matches = (valid_density['count'] / COUNT_DENSITY_DIVISOR).round(2).eq(valid_density['density'])
print(f'Count/3,1273 cocok dengan Density: {density_matches.sum()}/{len(valid_density)} baris')
print('Nilai kolom Area sumber:', history['area_km2'].dropna().unique().tolist())
display(registry[['corridor', 'tower_name', 'geometry_source_sheet', 'geometry_source_row', 'elevation_m', 'tower_height_m']].head(8))
print('Geometri kosong:', registry[['elevation_m', 'tower_height_m', 'span_left_m', 'span_right_m']].isna().sum().to_dict())
assert density_matches.all()

## 4. Eksplorasi setelah cleaning
### 4.1 Sebaran menurut tahun dan koridor
Grafik berikut menggambarkan variasi data; perbedaan kelompok tidak langsung berarti hubungan sebab-akibat.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history.boxplot(column='count', by='year', ax=axes[0], grid=False)
axes[0].set(title='Count petir per tahun', xlabel='Tahun', ylabel='Count')
history.boxplot(column='count', by='corridor', ax=axes[1], grid=False)
axes[1].set(title='Count petir per koridor', xlabel='Koridor', ylabel='Count')
fig.suptitle('')
plt.tight_layout()
plt.show()

## 5. Rekayasa fitur dan pemisahan waktu
### 5.1 Satu pasangan tahun
Untuk target tahun `t+1`, fitur hanya boleh memakai informasi yang tersedia sampai `t`. Tahun 2020 menjadi riwayat awal, bukan target tanpa data 2019. Nilai grounding snapshot 2026 ada untuk audit/sensitivitas, bukan fitur final historis.

### 5.2 Peran tahun
Target 2021–2024 adalah data pengembangan. Kandidat diuji berurutan pada 2023 dan 2024; target 2025 menjadi evaluasi retrospektif setelah pilihan dikunci.

In [ ]:
panel_result = build_model_panel(clean_path=clean.output_path, output_path=OUTPUT_DIR / 'ta_ml_model_panel.csv', overwrite=True)
panel = pd.read_csv(panel_result.output_path)
display(panel[['corridor', 'tower_key', 'feature_year', 'target_year', 'split', 'count_lag1', 'count_lag2', 'target_count']].head(8))
display(panel.groupby(['target_year', 'split']).size().rename('baris').reset_index())
assert len(panel) == TOWER_COUNT * 5 and (panel['target_year'] == panel['feature_year'] + 1).all()

## 6. Pemilihan algoritma dan kelompok fitur
### 6.1 Validasi berurutan
Putaran A: latih target 2021–2022, nilai 2023. Putaran B: latih 2021–2023, nilai 2024. Bandingkan dua algoritma × empat kelompok fitur dari rata-rata MAE kedua putaran; Poisson deviance menjadi pembeda berikutnya. Imputasi dan transformasi dipelajari dari data latih pada setiap putaran.

### 6.2 Latih ulang dan uji 2025
Setelah konfigurasi dipilih, latih ulang dengan target 2021–2024. `Count` nyata 2025 tidak ikut melatih model yang diuji pada 2025.

In [ ]:
evaluation = evaluate_models(panel_path=panel_result.output_path, pln_path=DATA_DIR / RAW_GEOMETRY_FILENAME, output_dir=OUTPUT_DIR, overwrite=True)
report = json.loads(evaluation.report_path.read_text(encoding='utf-8'))
predictions = pd.read_csv(evaluation.predictions_path)
candidates = pd.DataFrame(report['validation_candidates']['without_grounding'])
display(candidates[['model', 'feature_group', 'mean_mae', 'mean_poisson_deviance']].sort_values('mean_mae'))
print('Terpilih:', report['selected_model'], '+', report['selected_feature_group'])
print('Latih akhir:', report['final_fit_target_years'], '; uji:', report['final_test_year'])

## 7. Hasil evaluasi 2025
### 7.1 Kesalahan dan baseline
MAE adalah rata-rata selisih absolut antara ramalan dan `Count` nyata. Hasil 2025 pernah dilihat pada iterasi sebelumnya; skor baru ini **evaluasi retrospektif yang direvisi**, bukan tes buta yang baru dibuka sekali.

### 7.2 Sebaran kesalahan dan koridor
Periksa bukan hanya rata-rata seluruh tower, tetapi juga koridor dan tower yang sulit diperkirakan.

In [ ]:
display(pd.DataFrame({'model_terpilih': report['test_selected_model'], 'baseline_tahun_lalu': report['test_baseline_last_year']}))
display(pd.DataFrame(report['test_by_corridor']).T)
predictions['absolute_error'] = (predictions['prediction_ml'] - predictions['target_count']).abs()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(predictions['target_count'], predictions['prediction_ml'], s=12, alpha=0.55)
axes[0].set(xlabel='Count nyata 2025', ylabel='Prediksi ML', title='Prediksi vs pengamatan')
predictions.boxplot(column='absolute_error', by='corridor', ax=axes[1], grid=False)
axes[1].set(xlabel='Koridor', ylabel='Kesalahan absolut', title='Kesalahan per koridor')
fig.suptitle('')
plt.tight_layout()
plt.show()
display(predictions.nlargest(10, 'absolute_error')[['corridor', 'tower_key', 'target_count', 'prediction_ml', 'absolute_error']])

## 8. Keselarasan dengan perhitungan PLN
### 8.1 Samakan skala lebih dulu
Prediksi `Count` dinormalisasi ke `Ng`, lalu dikalikan collection area dan faktor eksposur PLN untuk mendapatkan prediksi pada skala `P_strike_annual`. Pembagi 3,1273 adalah hubungan empiris yang terverifikasi pada data sumber; kolom sumber `Area=2` tetap dicatat sebagai ketidaksesuaian definisi.

### 8.2 Cakupan dan pengecualian
Dua nilai `Ng` pada workbook jatuh ke default `8` karena format angka, sehingga tidak dipakai dalam ringkasan keselarasan. Nilai itu tetap terlihat pada file prediksi. PLN memakai petir 2025 yang teramati, jadi perbandingan ini tidak menyatakan PLN meramal 2025 dari data sebelumnya.

In [ ]:
alignment = report['pln_alignment']
display(pd.DataFrame([alignment]).drop(columns=['excluded_tower_keys', 'warning']))
print('Tower dikecualikan:', alignment['excluded_tower_keys'])
eligible = predictions[predictions['pln_ng_matches_observed_2025']]
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(eligible['p_strike_annual_pln'], eligible['prediction_p_strike_annual'], s=12, alpha=0.55)
ax.set(xlabel='P_strike_annual PLN', ylabel='Prediksi pada skala PLN', title='Keselarasan paparan tahunan')
plt.tight_layout()
plt.show()

## 9. Kesimpulan dan keterbatasan
### 9.1 Ringkasan yang diturunkan dari hasil
Hindari menyalin angka lama: sel berikut selalu mengambil metrik dari eksperimen yang baru dijalankan.

### 9.2 Batas interpretasi
Delapan riwayat 2020 belum terpetakan; atribut grounding berlabel 2026 tidak terbukti berlaku untuk seluruh tahun historis; geometri diperlakukan relatif tetap; keselarasan dengan PLN bukan validasi gangguan dua sirkit atau bukti sebab-akibat.

In [ ]:
print(f"Model: {report['selected_model']} + {report['selected_feature_group']}")
print(f"MAE 2025: {report['test_selected_model']['mae']:.2f}; baseline tahun lalu: {report['test_baseline_last_year']['mae']:.2f}")
print(f"Keselarasan PLN: {alignment['eligible_towers']}/617 tower; Spearman {alignment['spearman_rho']:.3f}")
print(report['test_protocol_note'])
print(alignment['warning'])